In [1]:
# 1) Instalar numpy compatible con torch + ultralytics
# !pip install -U "numpy<2" --force-reinstall

# 2) Reinstalar ultralytics SIN seguir dependencias (no actualiza numpy)
# !pip install ultralytics==8.3.0 --no-deps --quiet

# 3) Instalar deps compatibles
# !pip install "opencv-python<4.12" pillow matplotlib --quiet
# !pip install onnx==1.16.0 onnxruntime==1.18.0 onnxscript==0.1.0 --quiet
# !pip install gdown --quiet

In [15]:
import os
import zipfile
import gdown
import shutil
import torch
from ultralytics import YOLO
from datetime import datetime
import pickle
import requests
import json
import numpy as np
import math
import yaml
from google.colab import files
from pathlib import Path

In [3]:
data_dir = os.path.join(os.getcwd(), "data")

def descargar_dataset(data_dir):

  # Verificar si exsite la carpeta
  if not os.path.exists(data_dir):

    try:
      # Nombre del archivo ZIP que se va a guardar
      ZIP_NAME = "kaggle-xray_baggage_scanner_anomaly_detection.zip"
      zip_path = os.path.join(data_dir, ZIP_NAME)

      # Crear carpeta data
      os.makedirs(data_dir, exist_ok=True)

      # Descargar el ZIP
      print("Descargando archivo zip ...")
      # Lo tomamos de Google Drive porque Kaggle requeire autenticación
      gdown.download(id="1IqPblTm7nmKFpHXtl4beopE_SajTBoI0", output=zip_path, quiet=False)
      print("Archivo zip descargado.")

      # Descomprimir el ZIP
      print("Descomprimiendo archivo ...")
      with zipfile.ZipFile(zip_path, "r") as zf:
          zf.extractall(data_dir)
      print("Archivo descomprimido.")
    except:
      # En caso de error, eliminar la carpeta creada
      shutil.rmtree(data_dir, ignore_errors=True)
      print("Ocurrió un error al descargar el dataset.")

  print(f"Dataset descargado en: '{data_dir}'")


# Descargar el dataset (solo si no existe la carpeta data)
descargar_dataset(data_dir)

Dataset descargado en: '/content/data'


In [4]:
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.9.0+cu126
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB


In [5]:
DATA_YAML = os.path.join(data_dir, "data.yaml")
BASE_WEIGHTS = "yolov8s.pt" if torch.cuda.is_available() else "yolov8n.pt"
IMG_SIZE = 416
EPOCHS   = 100 # TODO:JA 100
PATIENCE = math.ceil(EPOCHS / 4)

DEVICE = 0 if torch.cuda.is_available() else "cpu"  # autodetección
print(f"DATA: {DATA_YAML} | WEIGHTS: {BASE_WEIGHTS} | DEVICE: {DEVICE}")


DATA: /content/data/data.yaml | WEIGHTS: yolov8s.pt | DEVICE: 0


In [6]:
# ============================================
# 3) Entrenamiento (equivalente a CLI de README)
# ============================================
model = YOLO(BASE_WEIGHTS)

train_results = model.train(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    patience=PATIENCE,
    device=DEVICE,     # 0 en GPU, "cpu" si no hay GPU
    project="runs_yolo",
    name=f"y8n_{IMG_SIZE}e{EPOCHS}",
    cache=True,        # acelera IO en Colab
    verbose=True
)

best_path = model.trainer.best
print("Best weights:", best_path)

New https://pypi.org/project/ultralytics/8.3.231 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/data/data.yaml, epochs=100, time=None, patience=25, batch=16, imgsz=416, save=True, save_period=-1, cache=True, device=0, workers=8, project=runs_yolo, name=y8n_416e100, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fa

wandb: Currently logged in as: jose-aviani (jose-aviani-universidad-de-buenos-aires) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...
AMP: checks passed ✅


train: Scanning /content/data/train/labels.cache... 6181 images, 0 backgrounds, 0 corrupt: 100%|██████████| 6181/6181 [00:00<?, ?it/s]

WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.


...


100 epochs completed in 1.092 hours.
Optimizer stripped from runs_yolo/y8n_416e100/weights/last.pt, 19.9MB
Optimizer stripped from runs_yolo/y8n_416e100/weights/best.pt, 19.9MB

Validating runs_yolo/y8n_416e100/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
Model summary (fused): 186 layers, 9,829,599 parameters, 0 gradients


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 56/56 [00:07<00:00,  7.77it/s]


                   all       1766       1766      0.851      0.798      0.828      0.361
                     0        391        391       0.98      0.977      0.982       0.52
                     1        389        389      0.878      0.861      0.884      0.371
                     2        225        225      0.739      0.516      0.594      0.197
                     3        366        366      0.795      0.744      0.802      0.362
                     4        395        395      0.862      0.891      0.878      0.356
Speed: 0.1ms preprocess, 0.4ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs_yolo/y8n_416e100


lr/pg0,▃████▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
lr/pg1,▆███▇▇▇▇▇▇▇▇▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁
lr/pg2,▃████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁
metrics/mAP50(B),▁▂▃▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇███████████████
metrics/mAP50-95(B),▁▂▃▄▅▅▅▅▆▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██▇███████████
metrics/precision(B),▅▄▁▄▄▅▅▃▄▅▆▆▆▇▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇█▇▇███████
metrics/recall(B),▁▂▂▃▃▄▄▄▄▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██▇████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


Best weights: runs_yolo/y8n_416e100/weights/best.pt


In [7]:
# ============================================
# Validación (val)
# ============================================
metrics_val = model.val(
  data=DATA_YAML,
  imgsz=IMG_SIZE,
  device=DEVICE,
  split="val"
)
print("VAL metrics:", metrics_val)

Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
Model summary (fused): 186 layers, 9,829,599 parameters, 0 gradients


val: Scanning /content/data/valid/labels.cache... 1766 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1766/1766 [00:00<?, ?it/s]

WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.



val: Caching images (0.9GB RAM): 100%|██████████| 1766/1766 [00:00<00:00, 5260.38it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 111/111 [00:07<00:00, 13.95it/s]


                   all       1766       1766      0.849      0.802      0.831      0.361
                     0        391        391      0.979      0.977      0.982      0.518
                     1        389        389      0.872      0.864      0.883      0.374
                     2        225        225      0.738      0.527      0.598      0.199
                     3        366        366      0.795      0.752      0.803      0.359
                     4        395        395      0.859      0.891      0.888      0.357
Speed: 0.1ms preprocess, 0.8ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to runs_yolo/y8n_416e1002
VAL metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f5fda720500>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(

In [8]:
# ============================================
# Validación (test)
# ============================================
metrics_test = model.val(
  data=DATA_YAML,
  imgsz=IMG_SIZE,
  device=DEVICE,
  split="test"
)
print("TEST metrics:", metrics_test)

Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)


val: Scanning /content/data/test/labels.cache... 883 images, 0 backgrounds, 0 corrupt: 100%|██████████| 883/883 [00:00<?, ?it/s]

WARNING ⚠️ cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.



val: Caching images (0.4GB RAM): 100%|██████████| 883/883 [00:00<00:00, 5642.52it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 56/56 [00:04<00:00, 12.99it/s]


                   all        883        883      0.884      0.816      0.864       0.39
                     0        166        166      0.941      0.982      0.983      0.526
                     1        193        193      0.892      0.839       0.89      0.391
                     2        118        118      0.833      0.568      0.673      0.247
                     3        203        203      0.853      0.803      0.842      0.386
                     4        203        203      0.899      0.887      0.931      0.401
Speed: 0.1ms preprocess, 0.7ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs_yolo/y8n_416e1003
TEST metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f60dc55e600>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence

In [12]:
DATETIME_FILENAME = datetime.now().strftime("%Y-%m-%d--%H-%M")

In [13]:
# --- Evaluar y recolectamos todas las metricas relevantes ---

OUT_JSON      = "metrics_full.json"
DOWNLOAD_JSON = f"metrics_full__{DATETIME_FILENAME}.json"
EVAL_SPLITS   = ["val", "test"]  # Datasets sobre los que valuamos

def ga(obj, names, default=None):
    for n in names:
        if hasattr(obj, n):
            return getattr(obj, n)
    return default

def to_float(x):
    try: return float(x)
    except: return None

def to_list(a):
    try: return a.tolist()
    except: return None

def derive_metrics(P, R):
    P = np.array(P, dtype=float)
    R = np.array(R, dtype=float)
    if P.size == 0 or R.size == 0:
        return np.array([]), np.array([]), np.array([]), None, None, None
    F1  = (2*P*R)/(P+R+1e-12)
    F2  = (5*P*R)/(4*P+R+1e-12)
    MR  = 1.0 - R
    return F1, F2, MR, float(np.mean(F1)), float(np.mean(F2)), float(np.mean(MR))

model = YOLO(best_path)

out = {
    "config": {
        "data_yaml": os.path.abspath(DATA_YAML),
        "weights": os.path.abspath(best_path),
        "imgsz": IMG_SIZE,
        "device": str(DEVICE),
    },
    "splits": {}
}

for split in EVAL_SPLITS:
    print(f"\n=== Eval {split} ===")
    m = model.val(data=DATA_YAML, imgsz=IMG_SIZE, device=DEVICE, split=split, plots=False, save_json=False)

    box = m.box
    # Global (Ultralytics a veces da mp/mr/map/map50; si no, promediamos por-clase)
    mp    = ga(box, ["mp","p"], None)
    mr    = ga(box, ["mr","r"], None)
    map50 = ga(box, ["map50","ap50"], None)
    map95 = ga(box, ["map","ap"], None)

    # Por-clase
    P    = np.array(ga(box, ["p_class","p"], []), dtype=float)
    R    = np.array(ga(box, ["r_class","r"], []), dtype=float)
    AP50 = np.array(ga(box, ["ap50_class","ap50"], []), dtype=float)
    AP   = np.array(ga(box, ["ap_class","ap"], []), dtype=float)

    names = getattr(m, "names", None) or getattr(model, "names", None) or [f"cls{i}" for i in range(len(P))]
    K = len(names)

    # Si no hay globales, calcular como promedios por clase
    if mp is None and P.size:
        mp = float(np.mean(P))
    if mr is None and R.size:
        mr = float(np.mean(R))
    if map50 is None and AP50.size:
        map50 = float(np.mean(AP50))
    if map95 is None and AP.size:
        map95 = float(np.mean(AP))

    # Derivadas
    F1, F2, MR, macro_F1, macro_F2, macro_MR = derive_metrics(P, R)

    # Speed y FPS
    speed = getattr(m, "speed", {}) or {}
    FPS = None
    try:
        total_ms = float(speed.get("preprocess", 0)) + float(speed.get("inference", 0)) + float(speed.get("postprocess", 0))
        FPS = 1000.0/total_ms if total_ms > 0 else None
    except:
        pass

    # Matriz de confusión (si existe)
    cm_raw = None
    cm_norm = None
    try:
        cm_obj = m.confusion_matrix
        mat = getattr(cm_obj, "matrix", None)
        if mat is not None:
            cm_raw = np.array(mat, dtype=float)
            col = cm_raw.sum(axis=0, keepdims=True); col[col==0]=1
            cm_norm = cm_raw/col
    except:
        pass

    out["splits"][split] = {
        "global": {
            "precision_macro": to_float(mp),
            "recall_macro": to_float(mr),
            "mAP50": to_float(map50),
            "mAP50_95": to_float(map95),
            "macro_F1": to_float(macro_F1),
            "macro_F2": to_float(macro_F2),
            "macro_miss_rate": to_float(macro_MR),
            "speed_ms_per_image": {k: to_float(v) for k, v in (speed.items() if isinstance(speed, dict) else [])},
            "FPS": to_float(FPS),
        },
        "per_class": [
            {
                "id": int(i),
                "name": str(names[i]),
                "P": to_float(P[i]) if i < len(P) else None,
                "R": to_float(R[i]) if i < len(R) else None,
                "AP50": to_float(AP50[i]) if i < len(AP50) else None,
                "AP": to_float(AP[i]) if i < len(AP) else None,
                "F1": to_float(F1[i]) if i < len(F1) else None,
                "F2": to_float(F2[i]) if i < len(F2) else None,
                "miss_rate": to_float(MR[i]) if i < len(MR) else None,
            } for i in range(K)
        ],
        "confusion_matrix": {
            "raw": to_list(cm_raw),
            "normalized_true": to_list(cm_norm)
        }
    }

# Export JSON
with open(OUT_JSON, "w") as f:
    json.dump(out, f, indent=2)

# Descargar metricas
shutil.copy(OUT_JSON, DOWNLOAD_JSON)
files.download(DOWNLOAD_JSON)

print(f"\n✅ Métricas exportadas en '{OUT_JSON}' y descargadas en '{DOWNLOAD_JSON}'")


=== Eval val ===
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
Model summary (fused): 186 layers, 9,829,599 parameters, 0 gradients


val: Scanning /content/data/valid/labels.cache... 1766 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1766/1766 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 111/111 [00:06<00:00, 18.14it/s]


                   all       1766       1766      0.849      0.802      0.831      0.361
                     0        391        391      0.979      0.977      0.982      0.518
                     1        389        389      0.872      0.864      0.883      0.374
                     2        225        225      0.738      0.527      0.598      0.199
                     3        366        366      0.795      0.752      0.803      0.359
                     4        395        395      0.859      0.891      0.888      0.357
Speed: 0.1ms preprocess, 0.6ms inference, 0.0ms loss, 0.8ms postprocess per image

=== Eval test ===
Ultralytics 8.3.0 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)


val: Scanning /content/data/test/labels.cache... 883 images, 0 backgrounds, 0 corrupt: 100%|██████████| 883/883 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 56/56 [00:03<00:00, 17.78it/s]

                   all        883        883      0.884      0.816      0.864       0.39
                     0        166        166      0.941      0.982      0.983      0.526
                     1        193        193      0.892      0.839       0.89      0.391
                     2        118        118      0.833      0.568      0.673      0.247
                     3        203        203      0.853      0.803      0.842      0.386
                     4        203        203      0.899      0.887      0.931      0.401
Speed: 0.1ms preprocess, 0.6ms inference, 0.0ms loss, 0.8ms postprocess per image


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Métricas exportadas en 'metrics_full.json' y descargadas en 'metrics_full__2025-11-24--22-15.json'


Mejor modelo:

In [10]:
best_path

PosixPath('runs_yolo/y8n_416e100/weights/best.pt')

In [17]:
# Hacemos download de los resultados YOLO

FOLDER_TO_ZIP = Path(best_path).parent.parent
ZIP_NAME = f"runs_yolo_best__{DATETIME_FILENAME}"
# 1) Crear el zip
shutil.make_archive(ZIP_NAME, 'zip', FOLDER_TO_ZIP)
# 2) Descargar el zip
files.download(f"{ZIP_NAME}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>